# Mini-Project Sprint: Multi-Agent Travel Planner

**Prompt Engineering & Autonomous Reasoning**

This practical builds a complete manager-worker travel-planning system.

Earlier in this module, you practised prompt structure, ReAct-style tool use, and Reflexion-style critique. This sprint turns those techniques into one inspectable project:

- Prompt contracts turn vague tasks into structured inputs and outputs.
- ReAct gives each worker an inspectable reason â†’ action â†’ observation â†’ decision trace.
- Reflexion lets the system critique and improve the first draft.
- Local tools simulate external travel APIs so the session can focus on orchestration.


## Practical outcome

You will run and inspect a travel-planner agent team:

1. A manager extracts requirements from a travel request.
2. Flight, hotel, and activity workers use local tools.
3. The manager synthesises a first itinerary.
4. A critic checks the itinerary.
5. The manager returns a refined final plan.

In [1]:
# TRAINER NOTE: Run this first to confirm the notebook is being executed from the project root.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

if not (PROJECT_ROOT / "main.py").exists():
    raise RuntimeError("Open this notebook from the mini_project_sprint project folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Project root: c:\Users\Navya\OneDrive\Desktop\vscode\mini_project_sprint


## Setup and runtime mode

The project supports two modes:

- `offline`: deterministic classroom demo with no API key required.
- `live`: real Groq calls for requirement extraction, worker ranking, synthesis, critique, and final refinement.

Start in `offline` mode first. Switch to `live` only after the class understands the architecture.

In [2]:
from utils import load_project_env, pretty_json

settings = load_project_env()
print(pretty_json(settings))

# COST NOTE: Offline mode costs â‚¹0. Live mode uses several compact openai/gpt-oss-120b calls.

{
  "demo_mode": "offline",
  "model_name": "openai/gpt-oss-120b",
  "temperature": "0.2",
  "langsmith_tracing": "false"
}


## Inspect the local travel tools

The tools simulate travel APIs. They are intentionally deterministic so the source of each fact is visible.

In [3]:
from tools.travel_tools import load_travel_data

travel_data = load_travel_data()
print("Flights:", len(travel_data["flights"]))
print("Hotels:", len(travel_data["hotels"]))
print("Activities:", len(travel_data["activities"]))

print("\nSample flight:")
print(pretty_json(travel_data["flights"][0]))

Flights: 6
Hotels: 5
Activities: 7

Sample flight:
{
  "flight_id": "SQ-421",
  "origin": "Mumbai",
  "destination": "Singapore",
  "airline": "Singapore Airlines",
  "depart_time": "11:45",
  "arrive_time": "19:50",
  "duration_hours": 5.6,
  "stops": 0,
  "price_inr": 28500,
  "baggage": "25 kg checked + 7 kg cabin",
  "tags": [
    "direct",
    "daytime",
    "premium",
    "reliable"
  ],
  "notes": "Comfortable direct daytime option; arrives before dinner."
}


## Step 1 â€” Start with a realistic user request

The request includes hard constraints and preferences. The manager must separate them before delegating work.

In [4]:
user_request = (
    "Plan a 4-day trip from Mumbai to Singapore for two people. Keep it mid-budget, "
    "avoid red-eye flights, prefer food, city views, and cultural sites. "
    "Keep the total hotel budget under â‚¹45,000."
)

print(user_request)

Plan a 4-day trip from Mumbai to Singapore for two people. Keep it mid-budget, avoid red-eye flights, prefer food, city views, and cultural sites. Keep the total hotel budget under â‚¹45,000.


## Step 2 â€” Create the manager

The manager owns orchestration. It does not directly search flights or hotels; it delegates to workers.

In [5]:
from utils import LLMClient
from agents.manager import TravelPlannerManager

llm = LLMClient(
    demo_mode=settings["demo_mode"],
    model_name=settings["model_name"],
    temperature=float(settings["temperature"]),
)

manager = TravelPlannerManager(llm=llm, travel_data=travel_data)
print("Manager ready in", settings["demo_mode"], "mode")

Manager ready in offline mode


## Step 3 â€” Manager extracts requirements

This is the first prompt contract. Free text becomes a structured object that every worker can consume.

In [6]:
requirements = manager.parse_request(user_request)
print(pretty_json(requirements))

{
  "origin": "Mumbai",
  "destination": "Singapore",
  "duration_days": 4,
  "traveller_count": 2,
  "budget_level": "mid-budget",
  "hotel_budget_inr": 45000,
  "avoid_red_eye": true,
  "preferences": [
    "food",
    "culture",
    "views"
  ],
  "assumptions": [
    "Local sample data simulates travel APIs.",
    "Prices are teaching estimates, not live market quotes.",
    "The final plan prioritises constraints before nice-to-have preferences."
  ],
  "missing_information": []
}


### What to notice

The manager should extract:

- route
- trip duration
- traveller count
- hotel budget
- hard constraints such as avoiding red-eye flights
- preferences such as food, views, and culture

If the extraction is weak, every downstream worker becomes less reliable.

## Step 4 â€” Run the workers individually

Each worker uses the same pattern:

`Thought â†’ Action â†’ Observation â†’ Decision`

This is ReAct as an auditable worker trace, not an uncontrolled chain-of-thought dump.

In [7]:
flight_report = manager.flight_worker.run(requirements)
print(pretty_json(flight_report))

{
  "worker": "flight_search",
  "react_trace": [
    "Thought: Need flights that match route and hard constraints.",
    "Action: search_flights(origin, destination, avoid_red_eye)",
    "Observation: Found 3 candidate option(s).",
    "Decision: Recommend SQ-421 because it best balances constraints and cost."
  ],
  "options": [
    {
      "flight_id": "SQ-421",
      "origin": "Mumbai",
      "destination": "Singapore",
      "airline": "Singapore Airlines",
      "depart_time": "11:45",
      "arrive_time": "19:50",
      "duration_hours": 5.6,
      "stops": 0,
      "price_inr": 28500,
      "baggage": "25 kg checked + 7 kg cabin",
      "tags": [
        "direct",
        "daytime",
        "premium",
        "reliable"
      ],
      "notes": "Comfortable direct daytime option; arrives before dinner.",
      "selection_score": 6
    },
    {
      "flight_id": "6E-101",
      "origin": "Mumbai",
      "destination": "Singapore",
      "airline": "IndiGo",
      "depart_time": 

In [8]:
hotel_report = manager.hotel_worker.run(requirements)
print(pretty_json(hotel_report))

{
  "worker": "hotel_search",
  "react_trace": [
    "Thought: Need hotel options matching destination, budget, and preferences.",
    "Action: search_hotels(destination, hotel_budget, preferences)",
    "Observation: Found 3 candidate option(s).",
    "Decision: Recommend Bugis Heritage Inn as the within budget."
  ],
  "options": [
    {
      "hotel_id": "SG-H03",
      "destination": "Singapore",
      "name": "Bugis Heritage Inn",
      "area": "Bugis",
      "price_per_night_inr": 8900,
      "rating": 4.1,
      "amenities": [
        "metro nearby",
        "self check-in"
      ],
      "tags": [
        "budget",
        "culture",
        "metro",
        "walkable"
      ],
      "notes": "Strong budget option near Arab Street and Kampong Glam.",
      "nights": 3,
      "total_price_inr": 26700,
      "selection_score": 18
    },
    {
      "hotel_id": "SG-H01",
      "destination": "Singapore",
      "name": "Harbour View City Hotel",
      "area": "Tanjong Pagar",
     

In [9]:
activity_report = manager.activity_worker.run(requirements)
print(pretty_json(activity_report))

{
  "worker": "activity_planner",
  "react_trace": [
    "Thought: Need activities that match trip preferences and fit across days.",
    "Action: search_activities(destination, preferences)",
    "Observation: Found 6 candidate activity option(s).",
    "Decision: Shortlist the strongest matches and leave room for pacing."
  ],
  "options": [
    {
      "activity_id": "SG-A04",
      "destination": "Singapore",
      "name": "Kampong Glam food and heritage trail",
      "area": "Bugis",
      "duration_hours": 3,
      "best_time": "Late afternoon",
      "cost_inr": 1500,
      "tags": [
        "food",
        "culture",
        "walkable",
        "low-cost"
      ],
      "notes": "Fits well with a Bugis hotel base.",
      "selection_score": 9
    },
    {
      "activity_id": "SG-A01",
      "destination": "Singapore",
      "name": "Chinatown and Maxwell Food Centre walk",
      "area": "Chinatown",
      "duration_hours": 3,
      "best_time": "Evening",
      "cost_inr": 160

## Step 5 â€” Combine worker outputs

The manager now has enough grounded information to build a first itinerary.

In [10]:
worker_outputs = {
    "flight": flight_report,
    "hotel": hotel_report,
    "activities": activity_report,
}

draft_itinerary = manager.create_draft_itinerary(requirements, worker_outputs)
print(pretty_json(draft_itinerary))

{
  "status": "draft",
  "selected_flight": {
    "flight_id": "SQ-421",
    "origin": "Mumbai",
    "destination": "Singapore",
    "airline": "Singapore Airlines",
    "depart_time": "11:45",
    "arrive_time": "19:50",
    "duration_hours": 5.6,
    "stops": 0,
    "price_inr": 28500,
    "baggage": "25 kg checked + 7 kg cabin",
    "tags": [
      "direct",
      "daytime",
      "premium",
      "reliable"
    ],
    "notes": "Comfortable direct daytime option; arrives before dinner.",
    "selection_score": 6
  },
  "selected_hotel": {
    "hotel_id": "SG-H03",
    "destination": "Singapore",
    "name": "Bugis Heritage Inn",
    "area": "Bugis",
    "price_per_night_inr": 8900,
    "rating": 4.1,
    "amenities": [
      "metro nearby",
      "self check-in"
    ],
    "tags": [
      "budget",
      "culture",
      "metro",
      "walkable"
    ],
    "notes": "Strong budget option near Arab Street and Kampong Glam.",
    "nights": 3,
    "total_price_inr": 26700,
    "selecti

### What to notice

The draft should be grounded in worker outputs:

- selected flight comes from local flight data
- selected hotel comes from local hotel data
- activities come from local activity data
- budget is calculated, not guessed

This separation is what makes the system debuggable.

## Step 6 â€” Reflexion critique

The critic checks the first draft for constraint violations, missing preferences, pacing issues, and unsupported assumptions.

In [11]:
critique = manager.critic.evaluate(requirements, draft_itinerary, worker_outputs)
print(pretty_json(critique))

{
  "overall_assessment": "Ready with minor polish",
  "issues": [
    {
      "severity": "low",
      "issue": "No major blocking issue found.",
      "fix": "Improve final answer by making assumptions and trade-offs explicit."
    }
  ],
  "revision_instructions": [
    "Improve final answer by making assumptions and trade-offs explicit."
  ],
  "quality_score": 90
}


## Step 7 â€” Final refinement

The final manager applies the critic's feedback and exposes what changed.

In [12]:
final_output = manager.refine_itinerary(requirements, draft_itinerary, critique, worker_outputs)
print(pretty_json(final_output))

{
  "final_itinerary": {
    "status": "final",
    "selected_flight": {
      "flight_id": "SQ-421",
      "origin": "Mumbai",
      "destination": "Singapore",
      "airline": "Singapore Airlines",
      "depart_time": "11:45",
      "arrive_time": "19:50",
      "duration_hours": 5.6,
      "stops": 0,
      "price_inr": 28500,
      "baggage": "25 kg checked + 7 kg cabin",
      "tags": [
        "direct",
        "daytime",
        "premium",
        "reliable"
      ],
      "notes": "Comfortable direct daytime option; arrives before dinner.",
      "selection_score": 6
    },
    "selected_hotel": {
      "hotel_id": "SG-H03",
      "destination": "Singapore",
      "name": "Bugis Heritage Inn",
      "area": "Bugis",
      "price_per_night_inr": 8900,
      "rating": 4.1,
      "amenities": [
        "metro nearby",
        "self check-in"
      ],
      "tags": [
        "budget",
        "culture",
        "metro",
        "walkable"
      ],
      "notes": "Strong budget op

## Step 8 â€” Run the full pipeline end-to-end

Now run the same process as one complete manager-worker-reflexion pipeline.

In [13]:
full_result = manager.run(user_request)
print(pretty_json(full_result["final_output"]))

{
  "final_itinerary": {
    "status": "final",
    "selected_flight": {
      "flight_id": "SQ-421",
      "origin": "Mumbai",
      "destination": "Singapore",
      "airline": "Singapore Airlines",
      "depart_time": "11:45",
      "arrive_time": "19:50",
      "duration_hours": 5.6,
      "stops": 0,
      "price_inr": 28500,
      "baggage": "25 kg checked + 7 kg cabin",
      "tags": [
        "direct",
        "daytime",
        "premium",
        "reliable"
      ],
      "notes": "Comfortable direct daytime option; arrives before dinner.",
      "selection_score": 6
    },
    "selected_hotel": {
      "hotel_id": "SG-H03",
      "destination": "Singapore",
      "name": "Bugis Heritage Inn",
      "area": "Bugis",
      "price_per_night_inr": 8900,
      "rating": 4.1,
      "amenities": [
        "metro nearby",
        "self check-in"
      ],
      "tags": [
        "budget",
        "culture",
        "metro",
        "walkable"
      ],
      "notes": "Strong budget op

## Debug view

When something goes wrong, inspect the state in this order:

1. Did requirement extraction capture the right constraints?
2. Did each worker return grounded options?
3. Did synthesis preserve the worker recommendations?
4. Did the critic find actionable issues?
5. Did the final manager actually revise the answer?

In [14]:
print("Requirement extraction:")
print(pretty_json(full_result["requirements"]))

print("\nCritique:")
print(pretty_json(full_result["critique"]))

Requirement extraction:
{
  "origin": "Mumbai",
  "destination": "Singapore",
  "duration_days": 4,
  "traveller_count": 2,
  "budget_level": "mid-budget",
  "hotel_budget_inr": 45000,
  "avoid_red_eye": true,
  "preferences": [
    "food",
    "culture",
    "views"
  ],
  "assumptions": [
    "Local sample data simulates travel APIs.",
    "Prices are teaching estimates, not live market quotes.",
    "The final plan prioritises constraints before nice-to-have preferences."
  ],
  "missing_information": []
}

Critique:
{
  "overall_assessment": "Ready with minor polish",
  "issues": [
    {
      "severity": "low",
      "issue": "No major blocking issue found.",
      "fix": "Improve final answer by making assumptions and trade-offs explicit."
    }
  ],
  "revision_instructions": [
    "Improve final answer by making assumptions and trade-offs explicit."
  ],
  "quality_score": 90
}


## Exercise 1 â€” Change the request

Try a different trip request using the same local data.

Hint: Mumbai â†’ Singapore and Mumbai â†’ Dubai are represented in the sample data.

In [15]:
# Try this:
# custom_request = "Plan a 3-day Dubai trip from Mumbai for two people. Prefer culture, food, and metro access."
# custom_result = manager.run(custom_request)
# print(pretty_json(custom_result["final_output"]))

## Exercise 2 â€” Add a worker improvement

Open `tools/travel_tools.py` and change one ranking rule.

Ideas:

- penalise early-morning flights
- prioritise metro access more strongly
- add a stronger penalty for hotels over budget

Run the same request before and after the change.

## Exercise 3 â€” Add a new critic rule

Open `agents/critic.py` and add one check.

Ideas:

- flag days with more than two major activities
- flag activities above a cost threshold
- require at least one meal-oriented activity for food-focused trips

## Closing summary

This sprint turns prompting techniques into a working agentic system:

- prompt contracts define reliable communication
- ReAct makes worker tool use inspectable
- manager-worker orchestration prevents one giant prompt from doing everything
- Reflexion improves the first draft before the final answer
- local tools keep the classroom build reliable while preserving the production architecture